# Notebook 01 — Prepare Dataset

**Project:** EM-guided Continual Learning for Chest X-ray Classification  
**Datasets:** CheXpert-v1.0-small and NIH ChestX-ray14  
**Purpose:** kiểm tra môi trường Kaggle, load dataset, tạo continual task stream và xuất thống kê ban đầu.

Notebook này chỉ kiểm tra dữ liệu. Chưa huấn luyện mô hình.

In [ ]:
import os
import sys
import platform
from pathlib import Path

import torch
import pandas as pd
import numpy as np

print("=" * 80)
print("Platform :", platform.platform())
print("Python   :", platform.python_version())
print("PyTorch  :", torch.__version__)
print("CUDA     :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU      :", torch.cuda.get_device_name(0))
else:
    print("GPU      : Not available")
print("=" * 80)

## 1. Clone hoặc cập nhật GitHub repo

Trên Kaggle, chạy cell này để lấy code mới nhất từ GitHub.

In [ ]:
REPO_URL = "https://github.com/thnguyenvn/em-cl-xray-poc.git"
PROJECT_ROOT = Path("/kaggle/working/em-cl-xray-poc")

if PROJECT_ROOT.exists():
    %cd /kaggle/working/em-cl-xray-poc
    !git pull
else:
    %cd /kaggle/working
    !git clone {REPO_URL}
    %cd /kaggle/working/em-cl-xray-poc

sys.path.append(str(PROJECT_ROOT))
print("PROJECT_ROOT:", PROJECT_ROOT)

## 2. Kiểm tra dataset đã gắn vào Kaggle Notebook

Cell này giúp xác định chính xác đường dẫn trong `/kaggle/input`.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")

print("Kaggle input folders:")
for p in sorted(INPUT_ROOT.glob("*")):
    print(" -", p)

print("\nPreview max depth 3:")
!find /kaggle/input -maxdepth 3 | head -200

## 3. Import các module dataset

Yêu cầu repo đã có các file trong `src/datasets/`.

In [ ]:
from src.datasets import (
    CheXpertDataset,
    NIHChestXrayDataset,
    get_train_transforms,
    get_eval_transforms,
    create_continual_task_stream,
    summarize_task_stream,
)

print("Dataset package loaded successfully.")

## 4. Cấu hình nhãn bệnh và task stream

PoC ban đầu dùng 5 nhãn để giảm độ phức tạp.

In [ ]:
TARGET_LABELS = [
    "No Finding",
    "Atelectasis",
    "Cardiomegaly",
    "Edema",
    "Pleural Effusion",
]

TASKS = [
    ["No Finding"],
    ["Atelectasis"],
    ["Cardiomegaly"],
    ["Edema"],
    ["Pleural Effusion"],
]

print("Target labels:", TARGET_LABELS)
print("Tasks:", TASKS)

## 5. Cấu hình đường dẫn dataset

Cần chỉnh lại `CHEXPERT_ROOT` và `NIH_ROOT` nếu tên Kaggle dataset khác.

In [ ]:
# ====== EDIT HERE IF NEEDED ======
CHEXPERT_ROOT = Path("/kaggle/input/chexpert-v10-small")
NIH_ROOT = Path("/kaggle/input/nih-chest-xrays")

CHEXPERT_TRAIN_CSV = CHEXPERT_ROOT / "train.csv"
CHEXPERT_VALID_CSV = CHEXPERT_ROOT / "valid.csv"
NIH_CSV = NIH_ROOT / "Data_Entry_2017.csv"

paths_to_check = {
    "CHEXPERT_ROOT": CHEXPERT_ROOT,
    "CHEXPERT_TRAIN_CSV": CHEXPERT_TRAIN_CSV,
    "CHEXPERT_VALID_CSV": CHEXPERT_VALID_CSV,
    "NIH_ROOT": NIH_ROOT,
    "NIH_CSV": NIH_CSV,
}

for name, path in paths_to_check.items():
    print(f"{name:22s}: {path} | exists={path.exists()}")

## 6. Nếu path sai, tự dò CSV quan trọng

In [ ]:
def find_file(filename, root="/kaggle/input"):
    matches = []
    for p in Path(root).rglob(filename):
        matches.append(p)
    return matches

for filename in ["train.csv", "valid.csv", "Data_Entry_2017.csv"]:
    matches = find_file(filename)
    print("\n", filename)
    if matches:
        for m in matches[:20]:
            print(" -", m)
    else:
        print(" - NOT FOUND")

## 7. Tạo transforms

Dùng ImageNet normalization vì backbone sau này sẽ dùng ResNet/DenseNet pretrained.

In [ ]:
IMAGE_SIZE = 224

train_transform = get_train_transforms(image_size=IMAGE_SIZE)
eval_transform = get_eval_transforms(image_size=IMAGE_SIZE)

print(train_transform)
print(eval_transform)

## 8. Load CheXpert

`uncertainty_policy="keep"` giữ nhãn `-1`, phục vụ thí nghiệm EM pseudo-label sau này.

In [ ]:
chexpert_train = None
chexpert_valid = None

if CHEXPERT_TRAIN_CSV.exists():
    chexpert_train = CheXpertDataset(
        root_dir=str(CHEXPERT_ROOT),
        csv_file=str(CHEXPERT_TRAIN_CSV),
        target_labels=TARGET_LABELS,
        transform=train_transform,
        uncertainty_policy="keep",
    )
    print("CheXpert train samples:", len(chexpert_train))
    display(chexpert_train.df[TARGET_LABELS].describe())
else:
    print("CheXpert train.csv not found. Please check CHEXPERT_ROOT.")

if CHEXPERT_VALID_CSV.exists():
    chexpert_valid = CheXpertDataset(
        root_dir=str(CHEXPERT_ROOT),
        csv_file=str(CHEXPERT_VALID_CSV),
        target_labels=TARGET_LABELS,
        transform=eval_transform,
        uncertainty_policy="keep",
    )
    print("CheXpert valid samples:", len(chexpert_valid))
else:
    print("CheXpert valid.csv not found. This is acceptable for initial PoC.")

## 9. Load NIH ChestX-ray14

In [ ]:
nih_dataset = None

if NIH_CSV.exists():
    nih_dataset = NIHChestXrayDataset(
        root_dir=str(NIH_ROOT),
        csv_file=str(NIH_CSV),
        target_labels=TARGET_LABELS,
        transform=eval_transform,
    )
    print("NIH samples:", len(nih_dataset))
    display(nih_dataset.df[TARGET_LABELS].describe())
else:
    print("NIH Data_Entry_2017.csv not found. Please check NIH_ROOT.")

## 10. Kiểm tra một sample ảnh

Nếu lỗi tại đây, thường là sai đường dẫn ảnh trong CSV hoặc cấu trúc thư mục Kaggle khác dự kiến.

In [ ]:
def inspect_sample(dataset, name="dataset", idx=0):
    if dataset is None:
        print(f"{name}: not loaded")
        return

    image, labels = dataset[idx]
    print(f"{name} sample {idx}")
    print("image type :", type(image))
    print("image shape:", tuple(image.shape) if hasattr(image, "shape") else "PIL")
    print("labels     :", labels)
    print("label names:", dataset.target_labels)

inspect_sample(chexpert_train, "CheXpert train", 0)
inspect_sample(nih_dataset, "NIH", 0)

## 11. Tạo Continual Learning task stream cho CheXpert

Dùng `require_single_positive=True` để PoC đơn giản hơn. Nếu số mẫu quá ít, đổi thành `False`.

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "outputs"
METRICS_DIR = OUTPUT_DIR / "metrics"
FIGURES_DIR = OUTPUT_DIR / "figures"

METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if chexpert_train is not None:
    chexpert_task_stream = create_continual_task_stream(
        dataset=chexpert_train,
        task_label_groups=TASKS,
        require_single_positive=True,
    )

    chexpert_summary = summarize_task_stream(chexpert_task_stream)
    display(chexpert_summary)

    chexpert_summary.to_csv(METRICS_DIR / "chexpert_task_summary.csv", index=False)
    print("Saved:", METRICS_DIR / "chexpert_task_summary.csv")
else:
    print("CheXpert train not loaded.")

## 12. Tạo Continual Learning task stream cho NIH

In [ ]:
if nih_dataset is not None:
    nih_task_stream = create_continual_task_stream(
        dataset=nih_dataset,
        task_label_groups=TASKS,
        require_single_positive=True,
    )

    nih_summary = summarize_task_stream(nih_task_stream)
    display(nih_summary)

    nih_summary.to_csv(METRICS_DIR / "nih_task_summary.csv", index=False)
    print("Saved:", METRICS_DIR / "nih_task_summary.csv")
else:
    print("NIH dataset not loaded.")

## 13. Thống kê nhãn nhanh

In [ ]:
def label_count_table(dataset, name):
    if dataset is None:
        return None
    df = dataset.df.copy()
    rows = []
    for label in dataset.target_labels:
        counts = df[label].value_counts(dropna=False).to_dict()
        row = {"dataset": name, "label": label}
        row.update({f"value_{k}": v for k, v in counts.items()})
        rows.append(row)
    return pd.DataFrame(rows)

tables = []
for ds, name in [(chexpert_train, "CheXpert_train"), (chexpert_valid, "CheXpert_valid"), (nih_dataset, "NIH")]:
    t = label_count_table(ds, name)
    if t is not None:
        tables.append(t)

if tables:
    label_summary = pd.concat(tables, ignore_index=True)
    display(label_summary)
    label_summary.to_csv(METRICS_DIR / "label_summary.csv", index=False)
    print("Saved:", METRICS_DIR / "label_summary.csv")

## 14. Kết luận kiểm tra dữ liệu

Nếu toàn bộ cell chạy được, Dataset Framework đã sẵn sàng cho Notebook 02: Fine-tuning Baseline.

In [ ]:
print("Notebook 01 completed.")
print("Next notebook: 02_finetune_baseline.ipynb")